<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/v3/gpu-systems/flash-attention-triton/flash-attention-triton.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("flash-attention-triton", ...)`


# Implement FlashAttention-2 in Triton

**Difficulty**: 🟣 Expert

**Companies**: NVIDIA, Meta, Together AI, xAI

---

### Problem Statement

Standard attention computes `softmax(Q @ K^T / sqrt(d)) @ V`, which requires **materializing the full N x N attention matrix**. For sequence length N=8192 and batch*heads=32, this matrix alone takes 8192² × 32 × 4 bytes ≈ 8.6 GB in float32. Standard attention is **O(N^2) in memory**, and because those N×N scores are written to and re-read from HBM, it is also memory-bandwidth bound.

**FlashAttention** (Dao et al., 2022) and its successor **FlashAttention-2** solve this by:
1. **Tiling**: Processing Q, K, V in blocks rather than all at once
2. **Online softmax**: Maintaining running softmax statistics (max and sum) across blocks
3. **Never materializing the full N x N matrix**: score tiles live in on-chip SRAM, so extra memory is O(N) instead of O(N^2), and HBM traffic drops sharply (the main source of the speedup)

The algorithm processes tiles of Q against ALL tiles of K and V, accumulating the attention output using the online softmax trick to correctly combine partial results from different K,V blocks. The result is **exact** attention, not an approximation.

What FlashAttention-2 changes relative to FlashAttention-1:
- **Loop order**: Q tiles in the outer loop, K/V tiles in the inner loop. Each Q tile's running statistics stay on-chip for the whole inner loop, and different Q tiles run in parallel (one program per (batch·head, Q tile)).
- **Deferred normalization**: keep the output *unnormalized* inside the loop (only rescale by `exp(m_old − m_new)`) and divide by the running sum `l` once at the end.
- **Causal masking**: skip K/V tiles that lie entirely in the future of the Q tile (about half the work), and mask only the tiles on the diagonal.

This notebook implements the **forward pass**. Autograd through the PyTorch loop below would store every P tile, which is O(N²) memory again. The real kernel saves only the output and the row-wise log-sum-exp `L = m + log(l)`, and its backward pass recomputes the tiles.

Your task:
1. Implement FlashAttention-2 in **pure PyTorch** (the tiled algorithm with online softmax), with optional causal masking
2. Write the Triton kernel version (runs on a GPU, or on CPU through Triton's interpreter)
3. Show that the output matches standard attention to float tolerance
4. Demonstrate the memory advantage

---

### Requirements

1. **Tiled Processing** — Split Q, and K/V, into tiles of `block_size` rows (one size for both here; real kernels tune `BLOCK_Q` and `BLOCK_KV` separately).
2. **Online Softmax** — For each Q block, iterate over all K,V blocks, maintaining `running_max` and `running_sum` to correctly combine softmax across K blocks.
3. **No Full Matrix** — Never create an N x N tensor. The largest intermediate should be `block_size_q x block_size_kv`.
4. **Correctness** — Output must match `torch.softmax(Q @ K^T / sqrt(d), dim=-1) @ V` within float32 tolerance.
5. **Causal masking** — `causal=True` must match `F.scaled_dot_product_attention(Q, K, V, is_causal=True)`. Skip K/V tiles that are entirely in the future, and mask positions `k > q` inside tiles that straddle the diagonal.

---

### Constraints

- ✅ Pure PyTorch implementation must work on CPU
- ✅ Triton kernel is optional: it needs CUDA, or `TRITON_INTERPRET=1` set before importing triton to run it on CPU (slow, but the same code)
- ✅ Must work for any sequence length (not just multiples of block_size)
- ❌ Do **not** materialize the full N x N attention matrix in the flash implementation

---

<details>
  <summary>💡 Hint</summary>

  **Online softmax across blocks:**
  
  When processing Q_block against K_block_j:
  ```
  S_j = Q_block @ K_block_j^T / sqrt(d)           # (block_q, block_kv)
  block_max_j = S_j.max(dim=-1, keepdim=True)      # max within this block
  new_max = max(running_max, block_max_j)           # global running max
  
  # Correction factor for previously accumulated results
  correction = exp(running_max - new_max)
  
  # Update running sum and output
  P_j = exp(S_j - new_max)                         # local attention weights
  running_sum = running_sum * correction + P_j.sum(dim=-1, keepdim=True)
  running_output = running_output * correction + P_j @ V_block_j
  running_max = new_max
  ```
  
  After all K,V blocks: `output = running_output / running_sum`

  **In the Triton kernel**, keys past the end of the sequence are loaded as zeros, so their scores come out as 0, not −∞. Mask them to `-inf` explicitly, or they leak into `running_sum` whenever N is not a multiple of the block size.

</details>

---

In [ ]:
import torch
import torch.nn.functional as F
import math
import os

# No GPU? Uncomment to run the Triton kernel on CPU with Triton's interpreter
# (must be set before triton is imported):
# os.environ["TRITON_INTERPRET"] = "1"

TRITON_AVAILABLE = False
try:
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
    print("Triton is available!")
except ImportError:
    print("Triton not available. Will use pure PyTorch implementation only.")

CUDA_AVAILABLE = torch.cuda.is_available()
print(f"CUDA available: {CUDA_AVAILABLE}")

# Triton kernels run on a GPU, or on CPU through Triton's interpreter
# (TRITON_INTERPRET=1 must be set before importing triton; slow, but handy for debugging).
TRITON_INTERPRET = os.environ.get("TRITON_INTERPRET") == "1"
RUN_TRITON = TRITON_AVAILABLE and (CUDA_AVAILABLE or TRITON_INTERPRET)
TRITON_DEVICE = "cuda" if CUDA_AVAILABLE else "cpu"

In [ ]:
# Test data
torch.manual_seed(42)

# Dimensions
batch_size = 2
n_heads = 4
seq_len = 128
head_dim = 64

# Create Q, K, V tensors: (batch, heads, seq_len, head_dim)
Q = torch.randn(batch_size, n_heads, seq_len, head_dim)
K = torch.randn(batch_size, n_heads, seq_len, head_dim)
V = torch.randn(batch_size, n_heads, seq_len, head_dim)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print(f"\nFull attention matrix would be: {batch_size} x {n_heads} x {seq_len} x {seq_len}")
print(f"= {batch_size * n_heads * seq_len * seq_len * 4 / 1024:.1f} KB")

In [ ]:
def standard_attention(Q, K, V):
    """
    Standard attention: O(N^2) memory.
    This is the reference implementation.
    
    Args:
        Q, K, V: (batch, heads, seq_len, head_dim)
    Returns:
        output: (batch, heads, seq_len, head_dim)
    """
    d = Q.shape[-1]
    # This creates the full N x N attention matrix
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d)  # (B, H, N, N)
    attn_weights = torch.softmax(scores, dim=-1)       # (B, H, N, N)
    output = attn_weights @ V                          # (B, H, N, d)
    return output

In [ ]:
def flash_attention_pytorch(Q, K, V, block_size=32, causal=False):
    """
    FlashAttention-2 forward pass in pure PyTorch.

    Processes attention in tiles, never materializing the full N x N matrix.
    Uses online softmax to correctly combine results across K,V blocks.

    Args:
        Q, K, V: (batch, heads, seq_len, head_dim)
        block_size: tile size for both Q and K,V blocks
        causal: if True, query i only attends to keys j <= i
    Returns:
        output: (batch, heads, seq_len, head_dim)
    """
    B, H, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    out_tiles = []

    # Process each Q tile
    for q_start in range(0, N, block_size):
        q_end = min(q_start + block_size, N)
        Q_block = Q[:, :, q_start:q_end, :]  # (B, H, block_q, d)

        # TODO: Initialize online softmax accumulators for this Q tile
        # (same dtype/device as Q):
        # running_max: (B, H, block_q, 1) = -inf, running_sum: (B, H, block_q, 1) = 0,
        # running_output: (B, H, block_q, d) = 0
        ...

        # TODO (causal): which K,V tiles can this Q tile skip entirely?
        for kv_start in range(0, N, block_size):
            kv_end = min(kv_start + block_size, N)
            K_block = K[:, :, kv_start:kv_end, :]  # (B, H, block_kv, d)
            V_block = V[:, :, kv_start:kv_end, :]  # (B, H, block_kv, d)

            # TODO: Compute local attention scores S: (B, H, block_q, block_kv)
            # TODO (causal): in a tile that straddles the diagonal, set S to -inf
            #                where the key position is after the query position
            ...

            # TODO: Online softmax update — new running max, rescale the previous
            # running_sum AND running_output by exp(old_max - new_max), then add this
            # tile's unnormalized weights P = exp(S - new_max) and P @ V_block
            ...

        # TODO: Normalize once, after the inner loop: running_output / running_sum,
        # and collect the tile's output
        ...

    return torch.cat(out_tiles, dim=2)

In [ ]:
if TRITON_AVAILABLE:
    @triton.jit
    def flash_attention_kernel(
        Q, K, V, Out,
        stride_qbh, stride_qn, stride_qd,
        stride_kbh, stride_kn, stride_kd,
        stride_vbh, stride_vn, stride_vd,
        stride_obh, stride_on, stride_od,
        N, scale,
        D: tl.constexpr,
        BLOCK_Q: tl.constexpr,
        BLOCK_KV: tl.constexpr,
        CAUSAL: tl.constexpr,
    ):
        """One program = one (batch*head, Q tile). It streams over the K/V tiles,
        keeping the running max m, running sum l and unnormalized output acc on-chip."""
        # TODO: Which Q tile and which (batch*head) does this program own? (tl.program_id)
        # TODO: Row/column offsets (tl.arange) and a mask for query rows past N;
        #       load the Q tile once
        # TODO: Initialize m (-inf), l (0) and acc (0) in float32
        # TODO: Loop over K/V tiles (causal: stop after the diagonal tile):
        #       - load K and V tiles, masking rows past N
        #       - scores = tl.dot(q, tl.trans(k)) * scale
        #       - keys past N were loaded as zeros: what must their score be?
        #         (and if CAUSAL, keys after the query)
        #       - online softmax update of m, l, acc (rescale by exp(m_old - m_new))
        # TODO: Normalize once and store the tile (masked)
        ...

    def flash_attention_triton(Q, K, V, block_size=32, causal=False):
        """Launch one program per (Q tile, batch*head). Q, K, V: (B, H, N, D) on CUDA,
        or on CPU under TRITON_INTERPRET=1."""
        # TODO: View (B, H, N, D) as (B*H, N, D), allocate the output,
        #       grid = (number of Q tiles, B*H), pass the strides, return (B, H, N, D)
        ...
else:
    print("Triton not available. Implement flash_attention_pytorch above;")
    print("pip install triton (and use a GPU or TRITON_INTERPRET=1) for the kernel.")

In [ ]:
# Validation
print("=" * 60)
print("VALIDATING FLASH ATTENTION")
print("=" * 60)

# Reference output
ref_output = standard_attention(Q, K, V)

# Flash attention output
flash_output = flash_attention_pytorch(Q, K, V, block_size=32)

# Test 1: Correctness
print("\n--- Correctness Test ---")
max_err = (flash_output - ref_output).abs().max().item()
print(f"Max absolute error: {max_err:.2e}")
assert torch.allclose(flash_output, ref_output, atol=1e-5), "Flash attention output MISMATCH!"
print("PASSED")

# Test 2: Different block sizes
print("\n--- Block Size Robustness Test ---")
for bs in [16, 32, 64]:
    flash_out = flash_attention_pytorch(Q, K, V, block_size=bs)
    err = (flash_out - ref_output).abs().max().item()
    assert torch.allclose(flash_out, ref_output, atol=1e-5), f"Failed for block_size={bs}"
    print(f"  block_size={bs}: max_err={err:.2e} PASSED")

# Test 3: Non-divisible sequence length
print("\n--- Non-Divisible Sequence Length Test ---")
Q2 = torch.randn(1, 1, 50, 32)
K2 = torch.randn(1, 1, 50, 32)
V2 = torch.randn(1, 1, 50, 32)
ref2 = standard_attention(Q2, K2, V2)
flash2 = flash_attention_pytorch(Q2, K2, V2, block_size=16)
err2 = (flash2 - ref2).abs().max().item()
assert torch.allclose(flash2, ref2, atol=1e-5), "Non-divisible seq len FAILED!"
print(f"  seq_len=50, block_size=16: max_err={err2:.2e} PASSED")

# Test 4: Causal masking
print("\n--- Causal Masking Test ---")
for (n, bs) in [(seq_len, 32), (50, 16), (50, 64)]:
    Qc, Kc, Vc = Q[:, :, :n], K[:, :, :n], V[:, :, :n]
    ref_c = F.scaled_dot_product_attention(Qc, Kc, Vc, is_causal=True)
    flash_c = flash_attention_pytorch(Qc, Kc, Vc, block_size=bs, causal=True)
    err_c = (flash_c - ref_c).abs().max().item()
    assert torch.allclose(flash_c, ref_c, atol=1e-5), f"Causal FAILED for seq_len={n}, block_size={bs}"
    print(f"  seq_len={n}, block_size={bs}: max_err={err_c:.2e} PASSED")

# Test 5: Triton kernel (GPU, or CPU via TRITON_INTERPRET=1)
print("\n--- Triton Kernel Test ---")
if RUN_TRITON:
    for (n, bs, causal) in [(seq_len, 32, False), (100, 32, False), (100, 32, True)]:
        q_t, k_t, v_t = (t[:, :, :n].to(TRITON_DEVICE) for t in (Q, K, V))
        ref_t = F.scaled_dot_product_attention(q_t, k_t, v_t, is_causal=causal)
        out_t = flash_attention_triton(q_t, k_t, v_t, block_size=bs, causal=causal)
        err_t = (out_t - ref_t).abs().max().item()
        # tl.dot may use TF32 on GPUs, hence the loose tolerance
        assert torch.allclose(out_t, ref_t, atol=1e-2), f"Triton kernel FAILED (N={n}, causal={causal})"
        print(f"  N={n}, block={bs}, causal={causal}: max_err={err_t:.2e} PASSED")
else:
    print("  Skipped (needs Triton plus CUDA or TRITON_INTERPRET=1)")

# Test 6: Memory comparison
print("\n--- Memory Analysis ---")
N = seq_len
d = head_dim
bs = 32
standard_mem = N * N  # full attention matrix
flash_mem_actual = bs * bs  # actual peak: one tile of scores
print(f"  Standard attention peak: {N}x{N} = {standard_mem} elements")
print(f"  Flash attention peak:    {bs}x{bs} = {flash_mem_actual} elements per tile (plus O(block_q * d) accumulators)")
print(f"  Memory reduction: {standard_mem / flash_mem_actual:.1f}x")

# For longer sequences, the advantage is even more dramatic
print("\n--- Scaling Analysis ---")
for N_test in [256, 1024, 4096, 8192]:
    standard = N_test * N_test * 4  # bytes in float32
    flash = bs * bs * 4
    print(f"  N={N_test:>5d}: standard={standard/1024/1024:>8.1f} MB, flash_tile={flash/1024:.1f} KB, ratio={standard // flash}x")

print("\nAll tests passed!")